# 02 — The main discontinuity test

For each bin, the count is predicted from its immediate neighbours:

$$\text{expected}_i = \frac{\text{count}_{i-1} + \text{count}_{i+1}}{2}$$

Under a smoothness null the difference is approximately normal, with the standard
deviation following the binomial argument used in this literature:

$$sd_i = \sqrt{N p_i (1-p_i) + \tfrac{1}{4} N (p_{i-1}+p_{i+1})(1 - p_{i-1} - p_{i+1})}$$

**Three commitments, fixed before looking at any output:**

1. Zero sits on a bin **boundary**, never inside a bin.
2. The test runs at **all three** bin widths (0.0025, 0.005, 0.01) and all three are
   reported. A result that appears at only one width is not a result.
3. Both the bin immediately **below** zero and the bin immediately **above** are
   reported. The hypothesis predicts a deficit below *and* a surplus above; finding
   only one is weaker evidence and is said so plainly.

In [1]:
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

np.random.seed(20250811)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

DATA = ROOT / "data"
FIGS = ROOT / "figures"
FIGS.mkdir(exist_ok=True)
print("project root:", ROOT)

project root: /Users/riddhi/Downloads/All Of My Claude Skills/FinanceDSProject


In [2]:
from src.binning import BIN_WIDTHS
from src.panel import restrict_window
from src.discontinuity import run_all_widths, run_test, summarize, chi_square_smooth_fit
from src.plotting import headline_histogram, bin_width_panel

panel = pd.read_parquet(DATA / "panel.parquet")
window, info = restrict_window(panel, "roa")

SOURCE = "SEC Financial Statement Data Sets (10-K filings)"
PERIOD = f"fiscal years {int(window.fiscal_year.min())}-{int(window.fiscal_year.max())}"
N_FIRMS = window["firm_id"].nunique()

print(f"panel:      {len(panel):,} firm-years after filters")
print(f"in window:  {info['inside_window']:,} firm-years ({N_FIRMS:,} firms), {PERIOD}")
print(f"outside |ROA| <= 0.10: {info['outside_window']:,} | no ROA computable: {info['missing_measure']:,}")

panel:      31,240 firm-years after filters
in window:  16,018 firm-years (3,582 firms), fiscal years 2016-2025
outside |ROA| <= 0.10: 15,222 | no ROA computable: 0


## The test at all three bin widths

In [3]:
results = run_all_widths(window["roa"], label="SEC 10-K, ROA")
cols = ["bin_width", "n_window", "count_below", "expected_below", "z_below", "p_below",
        "count_above", "expected_above", "z_above", "p_above"]
results[cols].round(4)

,bin_width,n_window,count_below,expected_below,z_below,p_below,count_above,expected_above,z_above,p_above
0,0.0025,16018,246,252.0,-0.3144,0.7532,291,252.5,1.9061,0.0566
1,0.0050,16018,459,479.5,-0.7913,0.4288,550,496.0,1.9540,0.0507
2,0.0100,16018,868,898.5,-0.8733,0.3825,1083,1017.0,1.7310,0.0834


In [4]:
print(summarize(results))

width=0.0025  N=16,018  below zero: 246 vs 252.0 expected (deficit, z=-0.31, p=0.753)  |  above zero: 291 vs 252.5 expected (surplus, z=+1.91, p=0.0566)
width=0.0050  N=16,018  below zero: 459 vs 479.5 expected (deficit, z=-0.79, p=0.429)  |  above zero: 550 vs 496.0 expected (surplus, z=+1.95, p=0.0507)
width=0.0100  N=16,018  below zero: 868 vs 898.5 expected (deficit, z=-0.87, p=0.383)  |  above zero: 1,083 vs 1017.0 expected (surplus, z=+1.73, p=0.0834)


### Goodness of fit across the whole window

Two versions. The first is the spec's chi-square against the neighbour-average
expectation; its degrees of freedom are approximate because the expectation is built
from the same counts. The second holds the two zero-adjacent bins *out* of a smooth
polynomial fit and then tests them against it, so its two degrees of freedom are
interpretable in the usual way.

In [5]:
rows = []
for w in BIN_WIDTHS:
    r = run_test(window["roa"], w)
    smooth = chi_square_smooth_fit(r["_table"])
    rows.append({
        "bin_width": w,
        "chi2_neighbour": r["chi2_neighbor"], "p_neighbour": r["chi2_neighbor_p"],
        "chi2_held_out": smooth["chi2"], "p_held_out": smooth["p_value"],
        "expected_below_fit": smooth["expected_below"],
        "expected_above_fit": smooth["expected_above"],
    })
pd.DataFrame(rows).round(4)

,bin_width,chi2_neighbour,p_neighbour,chi2_held_out,p_held_out,expected_below_fit,expected_above_fit
0,0.0025,114.5453,0.0036,13.9355,0.0009,229.8786,236.0248
1,0.0050,28.4175,0.8433,10.5228,0.0052,454.5279,479.1423
2,0.0100,22.2483,0.1754,10.8661,0.0044,882.3830,980.8803


## The headline figure

In [6]:
fig, table = headline_histogram(
    window["roa"], 0.005,
    source=SOURCE, n_firms=N_FIRMS, period=PERIOD,
    title="Distribution of scaled net income around zero",
    out=FIGS / "headline_histogram.png")
fig

<Figure size 1012x616 with 1 Axes>

### The same test at all three widths, side by side

In [7]:
fig = bin_width_panel(window["roa"], BIN_WIDTHS, source=SOURCE,
                      out=FIGS / "bin_width_panel.png")
fig

<Figure size 1518x484 with 3 Axes>

### Bin-level detail around zero

In [8]:
mid = len(table) // 2
detail = table[["left", "right", "count", "expected", "z", "p_value"]].iloc[mid - 5: mid + 5]
detail.round(4)

,left,right,count,expected,z,p_value
15,-0.025,-0.020,315,335.0,-0.9232,0.3559
16,-0.020,-0.015,356,336.5,0.8641,0.3875
17,-0.015,-0.010,358,382.5,-1.0621,0.2882
18,-0.010,-0.005,409,408.5,0.0205,0.9836
19,-0.005,0.000,459,479.5,-0.7913,0.4288
20,0.000,0.005,550,496.0,1.9540,0.0507
21,0.005,0.010,533,570.0,-1.3248,0.1852
22,0.010,0.015,590,554.5,1.2350,0.2168
23,0.015,0.020,576,592.5,-0.5729,0.5667
24,0.020,0.025,595,602.0,-0.2399,0.8104


In [9]:
results.to_csv(DATA / "main_test_results.csv", index=False)
table.to_csv(DATA / "headline_bin_table.csv", index=False)
print("wrote main_test_results.csv and headline_bin_table.csv")

wrote main_test_results.csv and headline_bin_table.csv


## Reading the result

Fill in from the tables above, not from expectation. The phrasing that is defensible
is **"consistent with earnings management"** — this test identifies a distributional
anomaly, and a distributional anomaly is not proof that managers manipulated
anything. The robustness checks in notebook 03, and the cash-flow placebo in
particular, are what decide how much weight the anomaly can carry.